# 不使用@tool的方式定義工具
## 1、舉例

In [5]:
# 配置模型
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
from rich import print as rprint

# 加载配置文件
load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GOOGLE_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

# 確保環境變數有 GOOGLE_API_KEY，或是直接在這裡把 key 傳進 api_key 參數
model = init_chat_model(
    model="gemma-4-31b-it",
    model_provider="openai",  # 關鍵：強制使用 OpenAI 的連線邏輯
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/" # 指向 Google
)

# 定義工具
def get_weather(city : str):
    return f"{city}天氣真好，天氣晴!!"

model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke('台北市')

rprint(response)

AIMessage(
    content='<thought>The user provided "台北市" (Taipei City).\nThis is a very brief input, essentially just a 
location.\nThe user might be looking for:\n1. General information about Taipei City.\n2. The current weather in 
Taipei City.\n3. Recommendations for things to do/eat in Taipei City.\n4. A translation or simple confirmation of 
the location.\n\nGiven that I have a `get_weather` tool, providing the current weather is a proactive and helpful 
response, but since the user didn\'t explicitly ask for it, I should probably provide a general overview or ask for
clarification while offering the weather.\n\nWait, looking at the prompt "台北市", it\'s a common way for users to 
ask for the status/info of a place.\n\nLet\'s check if I should just call the weather tool. Often, when a user just
types a city name, they want the weather. However, as a general-purpose AI, I should be more 
comprehensive.\n\nActually, the most helpful immediate action for a city name input is often to provide a brief 
summary and perhaps the weather.\n\nLet\'s try to be helpful:\n1. Acknowledge the city.\n2. Provide a brief 
description.\n3. Call the weather tool to give real-time info.\n\nPlan:\n1. Call `get_weather(city="Taipei")`.\n2. 
Respond with a general introduction to Taipei and the current weather.</thought>',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 17,
            'prompt_tokens': 35,
            'total_tokens': 355,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_provider': 'openai',
        'model_name': 'gemma-4-31b-it',
        'system_fingerprint': None,
        'id': 'HA_BarOlGcKA1e8PirKJ0Q4',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a10223-0867-7303-94d0-b8693b827809-0',
    tool_calls=[{'name': 'get_weather', 'args': {'city': 'Taipei'}, 'id': 'call_1189735', 'type': 'tool_call'}],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 35,
        'output_tokens': 17,
        'total_tokens': 355,
        'input_token_details': {},
        'output_token_details': {}
    }
)

# 2.工具描述的各部分詳解

## 2.1 了解convert_to_openai_tool

執行`model.bind_tools([get_weather])`，底層最終會調用`convert_to_openai_tool`生成工具描述。所以我們可以直接調用後者來查看解析工具描述

In [6]:
# 定義工具
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city : str):
    return f"{city}天氣真好，天氣晴!!"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}